## 0. 教程大纲

### 0.1 放大、缩小
```python
cv.resize(img, dsize, [interpolation])
```

### 0.2 平移变换
```python
M = np.array([[...]], dtype=np.float32)
cv.warpAffine(img, M, dsize)
```
### 0.3 错切变换

### 0.4 镜像变换
```python
cv.flip(img, 1) # 水平镜像
cv.flip(img, 0) # 垂直镜像
cv.flit(img, -1) # 水平垂直同时进行
```
### 0.5 旋转变换
```python
M = cv.getRotationMatrix2D(center, angle, scale)
img_rotate = cv.rotate(img, cv.ROTATE_90_CLOCKWISE)
```

### 0.6 透视变换
```python
M = cv.getPerspectiveTransform(src, dst)
img = cv.warpPerspective(img, M, dsize)
```

In [ ]:
import cv2 as cv              # OpenCV计算机视觉库
import numpy as np            # NumPy数值计算库
import matplotlib.pyplot as plt  # Matplotlib绑图库

In [ ]:
def show(img):
    """自定义显示函数：自动判断灰度图或彩色图并正确显示"""
    if img.ndim == 2:  # 灰度图，用gray色彩映射，vmin/vmax指定灰度范围
        plt.imshow(img, cmap='gray', vmin=0, vmax=255)
    else:  # 彩色图，BGR转RGB
        img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        plt.imshow(img)
    plt.show()

## 3 图像几何变换

In [ ]:
# cv.imread()读取图片，返回BGR格式的NumPy数组
img = cv.imread('pic/rabbit500x333.jpg')
show(img)

In [6]:
img.shape

(500, 333, 3)

In [ ]:
# 裁剪：通过NumPy数组切片提取感兴趣区域(ROI)
# img[行范围, 列范围, 通道]，取第150-450行、第50-300列、所有通道
rabbit = img[150:450, 50:300, :]
show(rabbit)

In [ ]:
# 仿射平移变换：定义2x3的仿射矩阵
# [[1, 0, tx], [0, 1, ty]] 表示向右平移tx像素、向下平移ty像素
transM = np.array([
    [1, 0, 20],   # x方向平移20像素
    [0, 1, 100]   # y方向平移100像素
], dtype=np.float32)

# cv.warpAffine()：应用仿射变换，参数为(图像, 仿射矩阵, 输出尺寸)
img_trans = cv.warpAffine(img, transM, dsize=(333, 500))
show(img_trans)

In [ ]:
# 保存平移前后的拼接图到文件
cv.imwrite("pic/img_rabbit_translate.jpg", np.hstack([img, img_trans]))

In [ ]:
# 错切变换（Shear）：沿x方向的错切
# [[1, shx, 0], [0, 1, 0]] 中shx=0.3表示x方向错切系数
shearM = np.array([
    [1, 0.3, 0],  # x方向错切，shx=0.3
    [0, 1,   0]
], dtype=np.float32)

# 输出尺寸需要调大以容纳错切后的图像
img_shear = cv.warpAffine(img, shearM, dsize=(400, 500))
show(img_shear)

In [ ]:
# 保存错切前后的拼接图
cv.imwrite("pic/img_rabbit_shear.jpg", np.hstack([img, img_shear]))

In [ ]:
# 镜像变换（水平翻转）：通过仿射矩阵实现
# [[-1, 0, width], [0, 1, 0]] 中-1表示x轴翻转，width是平移量使图像不超出画布
mirrorM = np.array([
    [-1, 0, 333],  # x轴翻转，平移333像素（图片宽度）到画布内
    [0,  1, 0]
], dtype=np.float32)

img_mirr = cv.warpAffine(img, mirrorM, dsize=img.shape[:2][::-1])  # shape[:2][::-1]获取(width, height)
show(img_mirr)

In [ ]:
# 保存镜像前后的拼接图
cv.imwrite("pic/img_rabbit_mirror.jpg", np.hstack([img, img_mirr]))

In [ ]:
# cv.flip()：OpenCV内置镜像函数
# flipCode=1：水平翻转；flipCode=0：垂直翻转；flipCode=-1：水平+垂直同时翻转
img_mirh = cv.flip(img, 1)   # 水平镜像（左右翻转）
img_mirv = cv.flip(img, 0)   # 垂直镜像（上下翻转）
img_mirb = cv.flip(img, -1)  # 水平和垂直同时翻转（旋转180度）

show(np.hstack([img, img_mirh, img_mirv, img_mirb]))  # 原图、水平、垂直、同时翻转

In [ ]:
# cv.rotate()：OpenCV内置90度倍数旋转函数
# ROTATE_90_CLOCKWISE：顺时针旋转90度（还有ROTATE_180、ROTATE_90_COUNTERCLOCKWISE）
img_rotate = cv.rotate(img, cv.ROTATE_90_CLOCKWISE)
show(img_rotate)

In [ ]:
# cv.getRotationMatrix2D()：获取旋转矩阵
# 参数：(旋转中心x, y, 旋转角度度数, 缩放比例)
# 旋转45度，缩放比例1（不缩放）
rotateM = cv.getRotationMatrix2D((80, 100), 45, 1)
img_rotate = cv.warpAffine(img, rotateM, dsize=(500, 500))
show(img_rotate)

In [ ]:
# 保存旋转前后的拼接图
cv.imwrite("pic/img_rabbit_rotate.jpg", np.hstack([img, img_rotate]))

In [ ]:
# 旋转+缩放：getRotationMatrix2D的第三个参数scale控制缩放比例
rotateM1 = cv.getRotationMatrix2D((80, 100), 45, 0.8)  # 缩小到80%
rotateM2 = cv.getRotationMatrix2D((80, 100), 45, 1)    # 不缩放
rotateM3 = cv.getRotationMatrix2D((80, 100), 45, 1.2)  # 放大到120%

img_rotate1 = cv.warpAffine(img, rotateM1, dsize=(700, 300))
img_rotate2 = cv.warpAffine(img, rotateM2, dsize=(700, 300))
img_rotate3 = cv.warpAffine(img, rotateM3, dsize=(700, 300))

# 并排显示：0.8倍 | 1倍 | 1.2倍
show(np.hstack([img_rotate1, img_rotate2, img_rotate3]))

In [17]:
print(img.shape)

(500, 333, 3)


In [ ]:
# cv.resize()：图像缩放，参数为(图像, (目标宽度, 目标高度))
# 默认使用双线性插值（INTER_LINEAR）
img_resize = cv.resize(img, (300, 200))
show(img_resize)

In [ ]:
##### 最近邻和双线性插值比较
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

def show(img):
    """显示函数"""
    plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB), cmap='gray', vmin=0, vmax=255)
    plt.show()

# 读取一张小图进行放大，比较不同插值方法的效果
img = cv.imread('pic/rabbit50x33.jpg')
# INTER_NEAREST：最近邻插值，速度快但有锯齿
img_resize1 = cv.resize(img, (330, 500), interpolation=cv.INTER_NEAREST)
# INTER_LINEAR：双线性插值，更平滑，OpenCV默认方法
img_resize2 = cv.resize(img, (330, 500), interpolation=cv.INTER_LINEAR)

show(img)
show(np.hstack([img_resize1, img_resize2]))  # 左：最近邻，右：双线性

### *3.10 最近邻采样

In [ ]:
# 用3x3矩阵演示双线性插值：将3x3放大为4x4
mat = np.array([
       [1, 2, 3],
       [4, 5, 6],
       [7, 8, 9]
], dtype=np.uint8)

# INTER_LINEAR：双线性插值，通过周围4个像素加权平均
mat2 = cv.resize(mat, (4, 4), interpolation=cv.INTER_LINEAR)
mat2

In [ ]:
# INTER_NEAREST：最近邻插值，直接取最近的像素值，不做加权平均
mat2 = cv.resize(mat, (4, 4), interpolation=cv.INTER_NEAREST)
mat2

In [ ]:
9/4

In [ ]:
(2,3) / (4/3) = (6/4, 9/4)
(1, 2) (1, 3)

(2, 2) (2, 3)

3/4 * (1/2 * 1 + 1/2 * 4)  + 1/4 * (1/2*2 + 1/2*5)

In [4]:
3/4 * (1/2 * 2 + 1/2 * 5)  + 1/4 * (1/2*3 + 1/2*6)

3.75

In [ ]:
(3,3) / (4/3) = (9/4, 9/4)
(2, 2) (2, 3)

(3, 2) (3, 3)

In [3]:
3/4 * (3/4 * 5 + 1/4 * 6)  + 1/4 * (3/4 * 8 + 1/4 * 9)

6.0

In [5]:
import cv2 as cv

In [ ]:
cv.getPerspectiveTransform()

In [ ]:
# 读取帕特农神庙图片用于透视变换演示
img = cv.imread('pic/parthenon500x750.jpg')
show(img)

In [ ]:
# 透视变换：定义4个源点和4个目标点
# src：原图中的四边形顶点（梯形的四个角）
# dst：目标矩形的四个顶点（校正为正矩形）
src = np.array([
    [210, 50],
    [610, 270],
    [650, 480],
    [150, 450]
], dtype=np.float32)

dst = np.array([
    [150, 50],
    [650, 50],
    [650, 480],
    [150, 480]
], dtype=np.float32)

# cv.getPerspectiveTransform()：根据源点和目标点计算3x3透视变换矩阵
M = cv.getPerspectiveTransform(src, dst)
M

In [ ]:
# cv.warpPerspective()：应用透视变换，参数为(图像, 3x3透视矩阵, 输出尺寸)
img2 = cv.warpPerspective(img, M, dsize=(750, 500))
show(img2)

In [20]:
import numpy as np

In [ ]:
# 保存透视变换前后的拼接图
cv.imwrite('pic/parthenon_perspective.jpg', np.hstack([img, img2]))

In [ ]:
# 读取带椒盐噪声的灰度图，用于滤波演示（预告下一章内容）
img2 = cv.imread('pic/rose_spnoise_200x200.jpg', 0)
show(img2)

In [ ]:
# cv.GaussianBlur()：高斯模糊滤波，去除噪声
# 参数：(图像, (核宽度, 核高度), 高斯核标准差sigmaX)
img3 = cv.GaussianBlur(img2, (5, 5), 10)
show(img3)

In [ ]:
# cv.bilateralFilter()：双边滤波，保边去噪
# 参数：(图像, 直径d, 颜色空间sigmaColor, 坐标空间sigmaSpace)
# 双边滤波在平滑噪声的同时保留边缘细节
img4 = cv.bilateralFilter(img2, 3, 10, 10)
show(img4)